# Run this notebook on Google Colab

In [1]:
!pip install optuna --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.4 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import random
import tensorflow as tf
from math import sqrt
import math

# Import Optuna for hyperparameter tuning
import optuna

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import GRU, Dropout, Dense
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Ensure TensorFlow is using GPU
device_name = tf.test.gpu_device_name()
if device_name:
    print(f"GPU Detected: {device_name}")
else:
    print("No GPU found. Training may be slow.")

GPU Detected: /device:GPU:0


In [5]:
# Train-val-test split (70% training, 20% validation, 10% test)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

assert math.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0), "Splits must sum to 1.0"

In [6]:
# Update dataset path for Google Drive
data_path = "/content/drive/My Drive/Colab Notebooks/4.0/Bitcoin_data_full.csv"
data = pd.read_csv(data_path, index_col=0, parse_dates=True) # Set index to datetime
data

,open,high,low,close,Volume BTC,Volume USD,SP500,CPI,Gold,VIX,Oil,fear_n_greed_index
date,,,,,,,,,,,,
2024-01-31 00:00:00,42945.0,42989.0,42933.0,42984.0,5.160732,221828.915894,4845.65,308.417,2037.1899,14.35,82.98,60.0
2024-01-31 00:01:00,42981.0,43056.0,42981.0,43056.0,7.969300,343126.169175,4845.65,308.417,2037.1899,14.35,82.98,60.0
2024-01-31 00:02:00,43056.0,43098.0,43022.0,43022.0,9.561501,411354.902045,4845.65,308.417,2037.1899,14.35,82.98,60.0
2024-01-31 00:03:00,43019.0,43029.0,43005.0,43005.0,0.598142,25723.117782,4845.65,308.417,2037.1899,14.35,82.98,60.0
2024-01-31 00:04:00,43002.0,43031.0,43002.0,43006.0,0.941927,40508.530194,4845.65,308.417,2037.1899,14.35,82.98,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 23:55:00,93476.0,93476.0,93471.0,93471.0,0.179799,16805.981112,5881.63,315.605,2623.8101,17.35,74.58,64.0
2024-12-31 23:56:00,93469.0,93469.0,93469.0,93469.0,0.021400,2000.236600,5881.63,315.605,2623.8101,17.35,74.58,64.0
2024-12-31 23:57:00,93462.0,93462.0,93413.0,93427.0,0.034750,3246.554616,5881.63,315.605,2623.8101,17.35,74.58,64.0


In [7]:
print("Missing values:\n", data.isna().sum())

Missing values:
 open                  0
high                  0
low                   0
close                 0
Volume BTC            0
Volume USD            0
SP500                 0
CPI                   0
Gold                  0
VIX                   0
Oil                   0
fear_n_greed_index    0
dtype: int64


In [8]:
# Identify target column index
target_col_idx = data.columns.get_loc('close')

In [9]:
# Train-val-test split
train_index = int(len(data) * TRAIN_RATIO)
val_index = train_index + int(len(data) * VAL_RATIO)

train_data = data.iloc[:train_index]
val_data = data.iloc[train_index:val_index]
test_data = data.iloc[val_index:]

print("Train shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Test shape:", test_data.shape)

Train shape: (327695, 12)
Validation shape: (93627, 12)
Test shape: (46815, 12)


In [10]:
# Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(train_data)

# Transform datasets
train_scaled = scaler.transform(train_data)
val_scaled = scaler.transform(val_data)
test_scaled = scaler.transform(test_data)

In [11]:
# Function to create sequences for time series forecasting
def create_sequences(data_array, window_size, target_col_idx):
    X, y = [], []
    for i in range(len(data_array) - window_size):
        X.append(data_array[i:(i + window_size), :])
        y.append(data_array[i + window_size, target_col_idx])
    return np.array(X), np.array(y)

In [12]:
# Function to inverse transform predictions
def inverse_transform_predictions(predicted_scaled, y_actual_scaled, scaler, train_data, target_col_idx):
    predicted_full = np.zeros((predicted_scaled.shape[0], train_data.shape[1]))
    y_actual_full = np.zeros((y_actual_scaled.shape[0], train_data.shape[1]))

    predicted_full[:, target_col_idx] = predicted_scaled[:, 0]
    y_actual_full[:, target_col_idx] = y_actual_scaled

    predicted_inverse = scaler.inverse_transform(predicted_full)[:, target_col_idx]
    y_actual_inverse = scaler.inverse_transform(y_actual_full)[:, target_col_idx]

    return predicted_inverse, y_actual_inverse

In [13]:
# Objective function for Optuna
def objective(trial):
    """
    Objective function for Optuna optimization.

    Returns:
    - Validation RMSE (Root Mean Squared Error).
    """
    # Suggest hyperparameters
    window_size = trial.suggest_int("WINDOW_SIZE", 30, 120, step=10)
    num_neurons = trial.suggest_int("NUM_NEURONS", 32, 128, step=16)
    dropout_rate = trial.suggest_float("DROPOUT_RATE", 0.1, 0.5, step=0.05)
    patience = trial.suggest_int("PATIENCE", 3, 10)
    epochs = trial.suggest_int("EPOCHS", 50, 200, step=10)
    batch_size = trial.suggest_categorical("BATCH_SIZE", [32, 64, 128, 256, 512])
    learning_rate = trial.suggest_loguniform("LEARNING_RATE", 1e-5, 1e-2)

    # Create sequences
    X_train, y_train = create_sequences(train_scaled, window_size, target_col_idx)
    X_val, y_val = create_sequences(val_scaled, window_size, target_col_idx)

    # Build the GRU model
    model = Sequential([
        GRU(num_neurons, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(dropout_rate),
        Dense(1)
    ])

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="mean_squared_error")

    # Early stopping
    early_stopping = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0)

    # Train the model with GPU support
    with tf.device('/device:GPU:0'):
        model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(X_val, y_val),
            callbacks=[early_stopping],
            verbose=0
        )

    # Predict on validation set
    y_val_pred_scaled = model.predict(X_val, verbose=0)

    # Inverse transform predictions
    predicted_val_inverse, y_val_inverse = inverse_transform_predictions(
        predicted_scaled=y_val_pred_scaled,
        y_actual_scaled=y_val,
        scaler=scaler,
        train_data=train_data,
        target_col_idx=target_col_idx
    )

    # Compute validation RMSE
    val_rmse = sqrt(mean_squared_error(y_val_inverse, predicted_val_inverse))

    return val_rmse  # Minimize RMSE

In [14]:
# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)  # Run for 20 trials

# Display best parameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

[I 2025-02-14 14:28:13,728] A new study created in memory with name: no-name-f2d63e79-2bd6-4c5c-8c38-8e3823a74de2
[I 2025-02-14 14:31:44,134] Trial 0 finished with value: 207.0111674396444 and parameters: {'WINDOW_SIZE': 50, 'NUM_NEURONS': 128, 'DROPOUT_RATE': 0.5, 'PATIENCE': 5, 'EPOCHS': 170, 'BATCH_SIZE': 64, 'LEARNING_RATE': 2.0458591463379097e-05}. Best is trial 0 with value: 207.0111674396444.
[I 2025-02-14 14:35:01,078] Trial 1 finished with value: 649.3767315772875 and parameters: {'WINDOW_SIZE': 60, 'NUM_NEURONS': 64, 'DROPOUT_RATE': 0.2, 'PATIENCE': 7, 'EPOCHS': 140, 'BATCH_SIZE': 128, 'LEARNING_RATE': 0.00030722490410408845}. Best is trial 0 with value: 207.0111674396444.
[I 2025-02-14 14:37:42,960] Trial 2 finished with value: 322.69501059160655 and parameters: {'WINDOW_SIZE': 110, 'NUM_NEURONS': 128, 'DROPOUT_RATE': 0.4, 'PATIENCE': 3, 'EPOCHS': 140, 'BATCH_SIZE': 64, 'LEARNING_RATE': 0.0002124113354033388}. Best is trial 0 with value: 207.0111674396444.
[I 2025-02-14 14:3

Best Hyperparameters: {'WINDOW_SIZE': 80, 'NUM_NEURONS': 112, 'DROPOUT_RATE': 0.1, 'PATIENCE': 5, 'EPOCHS': 70, 'BATCH_SIZE': 128, 'LEARNING_RATE': 0.0010536582863889035}
